# Crop Disease Detector — EfficientNet-B0 Training
PlantVillage Dataset | 38 Classes | Transfer Learning

In [ ]:
# STEP 0: Install dependencies (run this first if not installed)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'torch', 'torchvision', 'scikit-learn', 'matplotlib', 'pillow', 'tqdm', 'seaborn'],
    check=False)

In [ ]:
# Fix macOS SSL certificate errors that block model downloads
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import os
import json
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

print(f'PyTorch:     {torch.__version__}')
print(f'Torchvision: {torchvision.__version__}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device:      {DEVICE}')

# Safe num_workers for macOS notebooks
NUM_WORKERS = 0 if sys.platform == 'darwin' else 4
print(f'num_workers: {NUM_WORKERS}')

In [ ]:
# Load EfficientNet-B0 — handles both torchvision >= 0.13 and older versions
def load_efficientnet_b0(pretrained=True):
    tv_version = tuple(int(x) for x in torchvision.__version__.split('.')[:2])
    if tv_version >= (0, 13):
        from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
        print('Using new torchvision API (EfficientNet_B0_Weights)')
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        model = efficientnet_b0(weights=weights)
    else:
        from torchvision.models import efficientnet_b0
        print('Using legacy torchvision API (pretrained=True)')
        model = efficientnet_b0(pretrained=pretrained)
    return model

# Quick test — downloads weights here
print('Downloading pretrained EfficientNet-B0 weights...')
_test = load_efficientnet_b0(pretrained=True)
print('Model downloaded successfully!')
del _test

## 1. Data Loading & Augmentation

In [ ]:
import os

# Find the folder that directly contains disease class subfolders (e.g. Tomato___Early_blight)
def find_dataset_dir(base='/kaggle/input', max_depth=5):
    for root, dirs, files in os.walk(base):
        depth = root.replace(base, '').count(os.sep)
        if depth > max_depth:
            dirs.clear()
            continue
        disease_classes = [d for d in dirs if '___' in d]
        if len(disease_classes) >= 10:
            return root
    raise FileNotFoundError(
        f"No disease class folders found under {base}\n"
        f"Tree:\n" + '\n'.join(
            '  '*r.replace(base,'').count(os.sep) + os.path.basename(r) + '/'
            for r, ds, fs in os.walk(base)
            if r.replace(base,'').count(os.sep) <= 3
        )
    )

DATA_DIR = find_dataset_dir()
print(f'DATA_DIR = {DATA_DIR}')

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print('Loading dataset...')
base_dataset = datasets.ImageFolder(DATA_DIR)
class_names  = base_dataset.classes
num_classes  = len(class_names)
total        = len(base_dataset)
print(f'Total images : {total}')
print(f'Num classes  : {num_classes}')
for i, c in enumerate(class_names):
    print(f'  {i:02d}. {c}')

In [ ]:
# Save class names — goes to /kaggle/working/ (downloadable after run)
OUT_DIR = '/kaggle/working'
with open(f'{OUT_DIR}/class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)
print(f'Saved class_names.json → {OUT_DIR}/class_names.json')

In [ ]:
from torch.utils.data import Subset, Dataset

# Wrapper to apply different transforms to a subset without re-scanning disk
class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        # subset returns a PIL image because base_dataset has no transform
        return self.transform(img), label

# Deterministic 80/10/10 split
torch.manual_seed(42)
indices   = torch.randperm(total).tolist()
train_end = int(0.8 * total)
val_end   = int(0.9 * total)

train_subset = Subset(base_dataset, indices[:train_end])
val_subset   = Subset(base_dataset, indices[train_end:val_end])
test_subset  = Subset(base_dataset, indices[val_end:])

train_set = TransformSubset(train_subset, train_transforms)
val_set   = TransformSubset(val_subset,   val_transforms)
test_set  = TransformSubset(test_subset,  val_transforms)

BATCH_SIZE  = 32
NUM_WORKERS = 2  # Kaggle supports workers fine

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_set):,}  |  Val: {len(val_set):,}  |  Test: {len(test_set):,}')

## 2. Model — EfficientNet-B0 with Transfer Learning

In [ ]:
def build_model(num_classes, freeze_base=True):
    model = load_efficientnet_b0(pretrained=True)

    if freeze_base:
        for param in model.features.parameters():
            param.requires_grad = False

    # EfficientNet classifier: Sequential(Dropout, Linear)
    in_features = model.classifier[1].in_features  # 1280
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(in_features, num_classes)
    )
    return model

model = build_model(num_classes, freeze_base=True).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f'Trainable params : {trainable:,} / {total_p:,}')

## 3. Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = correct = total = 0
    for imgs, labels in tqdm(loader, leave=False, desc='train'):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            running_loss += loss.item() * imgs.size(0)
            correct += model(imgs).argmax(1).eq(labels).sum().item()
            total   += labels.size(0)
    return running_loss / total, correct / total


def eval_epoch(model, loader, criterion):
    model.eval()
    running_loss = correct = total = 0
    with torch.no_grad():
        for imgs, labels in tqdm(loader, leave=False, desc='eval '):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            running_loss += criterion(outputs, labels).item() * imgs.size(0)
            correct += outputs.argmax(1).eq(labels).sum().item()
            total   += labels.size(0)
    return running_loss / total, correct / total

In [ ]:
# ── Phase 1: Warm-up ── train classifier head only (5 epochs)
criterion  = nn.CrossEntropyLoss()
optimizer  = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_weights = None

print('=== Phase 1: Warm-up (head only, 5 epochs) ===')
for epoch in range(1, 6):
    tl, ta = train_epoch(model, train_loader, criterion, optimizer)
    vl, va = eval_epoch(model, val_loader, criterion)
    scheduler.step()
    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['val_loss'].append(vl);   history['val_acc'].append(va)
    print(f'Epoch {epoch:02d} | train loss {tl:.4f} acc {ta*100:.1f}% | val loss {vl:.4f} acc {va*100:.1f}%')
    if va > best_val_acc:
        best_val_acc  = va
        best_weights  = copy.deepcopy(model.state_dict())

print(f'\nBest val acc after warm-up: {best_val_acc*100:.2f}%')

In [ ]:
# ── Phase 2: Fine-tune ── unfreeze all, low LR (15 epochs)
for param in model.features.parameters():
    param.requires_grad = True

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-6)

print('=== Phase 2: Full Fine-tuning (15 epochs) ===')
for epoch in range(6, 21):
    tl, ta = train_epoch(model, train_loader, criterion, optimizer)
    vl, va = eval_epoch(model, val_loader, criterion)
    scheduler.step()
    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['val_loss'].append(vl);   history['val_acc'].append(va)
    marker = ' ***' if va > best_val_acc else ''
    print(f'Epoch {epoch:02d} | train loss {tl:.4f} acc {ta*100:.1f}% | val loss {vl:.4f} acc {va*100:.1f}%{marker}')
    if va > best_val_acc:
        best_val_acc = va
        best_weights = copy.deepcopy(model.state_dict())

print(f'\nFinal best val acc: {best_val_acc*100:.2f}%')

## 4. Save Best Model

In [ ]:
model.load_state_dict(best_weights)

checkpoint = {
    'model_state_dict': model.state_dict(),
    'class_names':      class_names,
    'num_classes':      num_classes,
    'val_accuracy':     best_val_acc,
}
torch.save(checkpoint, f'{OUT_DIR}/model.pt')
print(f'Saved model.pt → {OUT_DIR}/model.pt  (val acc: {best_val_acc*100:.2f}%)')
print()
print('Download both files from the Output tab on the right ->')
print('  model.pt and class_names.json')

## 5. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs, history['train_loss'], label='Train')
ax1.plot(epochs, history['val_loss'],   label='Val')
ax1.axvline(x=5.5, color='gray', linestyle='--', label='Unfreeze')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()

ax2.plot(epochs, [a*100 for a in history['train_acc']], label='Train')
ax2.plot(epochs, [a*100 for a in history['val_acc']],   label='Val')
ax2.axvline(x=5.5, color='gray', linestyle='--', label='Unfreeze')
ax2.axhline(y=90,  color='green', linestyle=':',  label='90% target')
ax2.set_title('Accuracy (%)'); ax2.set_xlabel('Epoch'); ax2.legend()

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/training_curves.png', dpi=120)
plt.show()
print(f'Best val accuracy: {best_val_acc*100:.2f}%')

## 6. Test Set Evaluation

In [ ]:
from sklearn.metrics import classification_report

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='testing'):
        imgs = imgs.to(DEVICE)
        preds = model(imgs).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f'Test Accuracy: {test_acc*100:.2f}%\n')

short_names = [c.split('___')[-1][:22] for c in class_names]
print(classification_report(all_labels, all_preds, target_names=short_names))